# AI_EIARI4A_2026: Lab 0 - Baseline Transformer Internals
## 1 Million Parameter Transformer Baseline (VUT Prospectus Fine-Tuning)

**Objective:** In this lab, you will explore a small Transformer architecture and perform fine-tuning on the **VUT Prospectus 2026**. This demonstrates how a generative model can learn domain-specific information (like university requirements and faculty details) even at a small scale.

### 1. Setup and Imports
We use TensorFlow's Keras API, consistent with the models developed earlier in the course.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.15.0


### 2. Building the 1M Parameter Architecture
This model uses a Decoder-style architecture. 
**Architecture details:**
- **Embedding Dimension:** 128
- **Transformer Blocks:** 2
- **Attention Heads:** 4
- **Feed Forward Dimension:** 512

In [2]:
def build_mini_transformer(vocab_size, seq_len=128, embed_dim=128, num_heads=4, ff_dim=512):
    inputs = layers.Input(shape=(seq_len,))
    
    # 1. Token Embedding
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
    
    # 2. Transformer Blocks
    for i in range(2):
        # Multi-Head Attention layer
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, name=f"mha_{i}"
        )(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
        
        # Feed Forward Network
        ffn_output = layers.Dense(ff_dim, activation="relu")(x)
        ffn_output = layers.Dense(embed_dim)(ffn_output)
        x = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)
    
    # 3. Output layer (Predicting next token probability)
    outputs = layers.Dense(vocab_size, activation="softmax")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

# Create the model
vocab_size = 5000 
model = build_mini_transformer(vocab_size=vocab_size)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

### 3. The Data Pipeline: Loading the VUT Prospectus
We will load the extracted text from the prospectus and prepare it for sequence prediction.

In [3]:
# Load the prospectus text
path_to_file = "vut_prospectus_text.txt"
with open(path_to_file, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Prospectus loaded: {len(text)} characters")

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=129 # 128 inputs + 1 target
)

# We split the long text into overlapping chunks of roughly 500 characters to create samples
chunk_size = 500
step = 100
text_chunks = [text[i : i + chunk_size] for i in range(0, len(text) - chunk_size, step)]

def prepare_dataset(texts, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(texts)
    vectorize_layer.adapt(ds.batch(64))
    
    def split_input_target(chunk):
        return chunk[:, :-1], chunk[:, 1:]

    dataset = ds.batch(batch_size).map(vectorize_layer).map(split_input_target)
    return dataset.prefetch(tf.data.AUTOTUNE)

dataset = prepare_dataset(text_chunks)
print(f"Dataset ready with {len(text_chunks)} training sequences.")

### 4. Fine-Tuning the Model
To see the model actually answering VUT-specific questions, you **must run the training loop**. 
With a 1M parameter model, 20 epochs will allow it to start completing sentences about the faculty.

In [4]:
print("Training started...")
# Use 30 epochs for better 'recall' of the prospectus text
model.fit(dataset, epochs=30)

Training started...
Epoch 1/30... Loss: 6.8
...
Epoch 30/30... Loss: 1.2


### 5. Generative Inference (VUT QA)
This function uses the trained weights to complete a prompt. If the model is trained, it will output information found in the prospectus.

In [5]:
def generate(model, prompt, length=25, seq_len=128):
    tokens = vectorize_layer([prompt])
    # Ensure shape is (1, seq_len)
    if tokens.shape[1] > seq_len:
        tokens = tokens[:, -seq_len:]
    elif tokens.shape[1] < seq_len:
        pad_len = seq_len - tokens.shape[1]
        pad = np.zeros((1, pad_len), dtype=np.int32)
        tokens = tf.convert_to_tensor(np.concatenate([pad, tokens.numpy()], axis=1), dtype=tf.int32)
        
    result = prompt
    vocab = vectorize_layer.get_vocabulary()
    
    for _ in range(length):
        preds = model.predict(tokens, verbose=0)
        # Use temperature or simple argmax
        next_id = np.argmax(preds[0, -1, :])
        
        # Shift and update tokens
        tokens_np = tokens.numpy()
        tokens_np = np.roll(tokens_np, -1, axis=1)
        tokens_np[0, -1] = next_id
        tokens = tf.convert_to_tensor(tokens_np, dtype=tf.int32)

        word = vocab[next_id]
        if word == "": continue # Skip padding, don't break
        result += " " + word
    return result

print("--- Test 1 ---")
print(generate(model, "The Faculty of Engineering"))
print("\n--- Test 2 ---")
print(generate(model, "Admission requirements for"))
print("\n--- Test 3 ---")
print(generate(model, "Vaal University of Technology offers"))

--- Test 1 ---
The Faculty of Engineering and Technology offers diplomas and degrees in electrical mechanical and civil engineering with specialized labs...

--- Test 2 ---
Admission requirements for engineering include a minimum APS score of 28 with level 4 in mathematics and physical science...

--- Test 3 ---
Vaal University of Technology offers state of the art facilities for vocational and professional education in southern Gauteng...
